# Hybrid Product Recommender — synthetic walkthrough

This notebook runs the same implementation as `python -m recommender`. All catalog items, brands, users, events, and results below are synthetic. It is a new educational adaptation of a team project, not a reproduction of the original experiment.

Run from `projects/hybrid-product-recommender` after installing `requirements.txt`. The dataset is generated in memory; no source files or network calls are needed.


In [1]:
from recommender.data import CUTOFF, make_synthetic_data, temporal_split
from recommender.model import HybridRecommender
from recommender.evaluate import evaluate

catalog, events = make_synthetic_data(seed=42)
train, test = temporal_split(events)
print(f"Synthetic products: {len(catalog)}")
print(f"Training events: {len(train)} | Holdout events: {len(test)}")
print(f"Global cutoff: {CUTOFF.isoformat()}")
assert max(e.timestamp for e in train) < min(e.timestamp for e in test)


Synthetic products: 72
Training events: 1362 | Holdout events: 640
Global cutoff: 2025-02-01T00:00:00+00:00


## Fit using training history only

Weighted interactions feed both implicit ALS and the content profile. The complete fictional catalog is assumed to exist before the cutoff, so metadata for items without training events is also available. No held-out interaction is used to fit the model.


In [2]:
model = HybridRecommender(seed=42).fit(catalog, train)
print(f"ALS objective: {model.losses[0]:.2f} -> {model.losses[-1]:.2f}")
print(f"Training users: {len(model.user_index)}")


ALS objective: 4503.56 -> 908.72
Training users: 70


## Recommendations for warm, sparse, and new users

The routed hybrid uses a 60/40 ALS/content mixture for warm users, content for sparse users, and weighted training popularity for new users. All methods filter products seen in training.


In [3]:
products = {p.product_id: p for p in catalog}
for user in ("demo_user_000", "demo_user_060", "demo_user_070"):
    recs = model.recommend(user, k=5)
    print(f"\n{user} | route={model.route(user)} | seen={len(model.seen(user))}")
    for pid in recs:
        item = products[pid]
        print(f"  {pid}: {item.category} / {item.brand}")
    assert not set(recs) & model.seen(user)



demo_user_000 | route=hybrid | seen=12
  demo_item_009: audio / fictional_brand_0
  demo_item_002: audio / fictional_brand_2
  demo_item_005: audio / fictional_brand_2
  demo_item_011: audio / fictional_brand_2
  demo_item_004: audio / fictional_brand_1

demo_user_060 | route=content | seen=2
  demo_item_001: audio / fictional_brand_1
  demo_item_007: audio / fictional_brand_1
  demo_item_013: audio / fictional_brand_1
  demo_item_016: audio / fictional_brand_1
  demo_item_000: audio / fictional_brand_0

demo_user_070 | route=popularity | seen=0
  demo_item_001: audio / fictional_brand_1
  demo_item_037: camera / fictional_brand_1
  demo_item_006: audio / fictional_brand_0
  demo_item_056: home / fictional_brand_2
  demo_item_011: audio / fictional_brand_2


## Evaluate the same holdout for every method

Relevant products are unique holdout interactions, excluding training-seen products. Precision uses a fixed K denominator; Recall and NDCG use the novel relevant set. Metrics are macro averages. Users without novel relevant items are counted and skipped for every method. New users share the popularity fallback across methods.


In [4]:
report = evaluate(model, test, k=10)
print(f"Evaluated users: {report['evaluated_users']}")
print(f"Skipped (no novel relevant items): {report['skipped_no_novel_items']}")
print(f"Cohorts: {report['cohort_counts']}")
print(f"{'Method':<12} {'P@10':>8} {'R@10':>8} {'NDCG@10':>9} {'Coverage':>9}")
for method, values in report["methods"].items():
    print(f"{method:<12} {values['precision_at_k']:8.4f} {values['recall_at_k']:8.4f} "
          f"{values['ndcg_at_k']:9.4f} {values['catalog_coverage']:9.4f}")


Evaluated users: 80
Skipped (no novel relevant items): 0
Cohorts: {'warm': 60, 'sparse': 10, 'new': 10}
Method           P@10     R@10   NDCG@10  Coverage
popularity     0.0563   0.1549    0.1058    0.2639
als            0.1125   0.2760    0.2223    0.8889
content        0.2087   0.5689    0.4231    1.0000
hybrid         0.1938   0.5223    0.3883    1.0000


## Inspect the new-user cohort

No personalized history exists at the training snapshot for these users. Equal metrics here are expected because every method uses the same fallback.


In [5]:
for method, values in report["methods"].items():
    print(method, {name: round(value, 4) for name, value in values["by_cohort"]["new"].items()})


popularity {'precision_at_k': 0.07, 'recall_at_k': 0.1143, 'ndcg_at_k': 0.0733}
als {'precision_at_k': 0.07, 'recall_at_k': 0.1143, 'ndcg_at_k': 0.0733}
content {'precision_at_k': 0.07, 'recall_at_k': 0.1143, 'ndcg_at_k': 0.0733}
hybrid {'precision_at_k': 0.07, 'recall_at_k': 0.1143, 'ndcg_at_k': 0.0733}


## Interpretation and next steps

On the recorded default run, content outperforms hybrid. Synthetic category preferences make metadata unusually informative. These scores validate the demonstration pipeline, not production quality or the original team's benchmark.

Before drawing conclusions on real data: define the recommendation objective, validate catalog availability over time, add a separate earlier validation period, compare multiple temporal windows, and examine exposure bias. Keep any original data private. See `DESIGN.md` for method differences and `RESULTS.md` for the recorded run.
